# DistilBERT 测试集评估

按 Notebook 019 的最低验证损失规则，使用种子 42 的第 1 轮检查点。固定分词器、256 词项上限、BF16 推理和类别分数的 argmax 规则。第 0 类为负面，第 1 类为正面。与此前六个方案使用同一份清洗后 IMDB 测试集。

这是预训练模型的扩展对照。项目中其他方案的测试结果已在本实验之前查看，DistilBERT 检查点则由验证损失选择。


In [ ]:
# 核对训练来源，冻结本次评估的文件与设置
import hashlib
import json
import math

import pandas as pd
import torch
import transformers
from sklearn.metrics import confusion_matrix
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.paths import CLEANED_DIR, CONFIGS_DIR, DATA_DIR, TABLES_DIR

def sha256(path):
    with path.open("rb") as file:
        return hashlib.file_digest(file, "sha256").hexdigest()

model_dir = DATA_DIR / "models" / "distilbert_seed42"
training_path = CONFIGS_DIR / "distilbert_finetuning.json"
previous_protocol_path = CONFIGS_DIR / "final_test_protocol.json"
training = json.loads(training_path.read_text(encoding="utf-8"))
previous_protocol = json.loads(previous_protocol_path.read_text(encoding="utf-8"))
checkpoint = json.loads((model_dir / "run_metadata.json").read_text(encoding="utf-8"))
history = pd.read_csv(TABLES_DIR / "distilbert_training_history.csv")
selected_row = history.loc[history["VALIDATION LOSS"].idxmin()]
validation_path = CLEANED_DIR / "val.csv"
test_path = CLEANED_DIR / "test.csv"
prior_results_path = TABLES_DIR / "fixed_experiment_summary.csv"
prior_predictions_path = TABLES_DIR / "fixed_models_test_predictions.csv"

if (sha256(training_path) != checkpoint["protocol_sha256"]
        or checkpoint["best_epoch"] != int(selected_row["EPOCH"])
        or checkpoint["best_epoch"] != 1):
    raise ValueError("Checkpoint does not match the selected training epoch.")
if sha256(validation_path) != training["split_sha256"]["validation"]:
    raise ValueError("Validation data differs from the fine-tuning run.")
test_hash = sha256(test_path)
if test_hash != previous_protocol["files"]["test"]["sha256"]:
    raise ValueError("Test data differs from the previous evaluation.")
if (training["precision"] != "bf16" or not torch.cuda.is_available()
        or not torch.cuda.is_bf16_supported()):
    raise RuntimeError("This fixed evaluation requires CUDA with BF16 support.")
if str(torch.__version__) != training["torch"] or transformers.__version__ != training["transformers"]:
    raise RuntimeError("Torch or Transformers version differs from the fine-tuning run.")

device = torch.device("cuda")
protocol_path = CONFIGS_DIR / "distilbert_test_protocol.json"
comparison_path = TABLES_DIR / "distilbert_test_comparison.csv"
prediction_path = TABLES_DIR / "distilbert_test_predictions.csv"
matrix_path = TABLES_DIR / "distilbert_test_confusion_matrix.csv"
if any(path.exists() for path in [comparison_path, prediction_path, matrix_path]):
    raise FileExistsError("Test results already exist. Inspect the saved tables.")
protocol = {
    "model_id": training["model_id"], "revision": training["revision"],
    "seed": training["seed"], "epoch": checkpoint["best_epoch"],
    "selection": training["selection"],
    "training_protocol_sha256": checkpoint["protocol_sha256"],
    "max_length": training["max_length"], "batch_size": training["batch_size"],
    "pad_to_multiple_of": 8, "precision": training["precision"],
    "attention_implementation": training["attention_implementation"],
    "torch": training["torch"], "transformers": training["transformers"],
    "decision_rule": "argmax over negative and positive logits",
    "test_sha256": test_hash, "prior_test_protocol_sha256": sha256(previous_protocol_path),
    "prior_comparison_sha256": sha256(prior_results_path),
    "prior_predictions_sha256": sha256(prior_predictions_path),
    "checkpoint_sha256": {
        name: sha256(model_dir / name)
        for name in ["model.safetensors", "config.json", "tokenizer.json", "tokenizer_config.json", "run_metadata.json"]
    },
}
protocol_text = json.dumps(protocol, ensure_ascii=False, indent=2) + chr(10)
if protocol_path.exists():
    if protocol_path.read_text(encoding="utf-8") != protocol_text:
        raise ValueError("A different DistilBERT test protocol already exists.")
else:
    protocol_path.write_text(protocol_text, encoding="utf-8")
print(f"Fixed test protocol: {sha256(protocol_path)}")


In [ ]:
# 先恢复已选检查点并核对验证指标
tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, trust_remote_code=False)
model = AutoModelForSequenceClassification.from_pretrained(
    model_dir, local_files_only=True, use_safetensors=True, trust_remote_code=False,
    attn_implementation=protocol["attention_implementation"],
).to(device)
model.eval()
if model.config.id2label != {0: "NEGATIVE", 1: "POSITIVE"}:
    raise ValueError("Checkpoint label mapping does not match the evaluation.")

def predict(data, name):
    labels = []
    predictions = []
    probabilities = []
    loss_sum = 0.0
    truncated = 0
    batch_size = protocol["batch_size"]
    with torch.inference_mode():
        for start in range(0, len(data), batch_size):
            rows = data.iloc[start:start + batch_size]
            texts = rows["text"].tolist()
            lengths = tokenizer(texts, truncation=False, return_length=True, verbose=False)["length"]
            truncated += sum(length > protocol["max_length"] for length in lengths)
            batch = tokenizer(
                texts, truncation=True, max_length=protocol["max_length"], padding=True,
                pad_to_multiple_of=protocol["pad_to_multiple_of"], return_tensors="pt",
                return_token_type_ids=False,
            ).to(device)
            batch_labels = torch.tensor(rows["label"].tolist(), dtype=torch.long, device=device)
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                output = model(**batch, labels=batch_labels)
            loss_sum += float(output.loss) * len(rows)
            labels.extend(batch_labels.cpu().tolist())
            predictions.extend(output.logits.argmax(dim=1).cpu().tolist())
            probabilities.extend(output.logits.float().softmax(dim=1)[:, 1].cpu().tolist())
            if start and start % (batch_size * 500) == 0:
                print(f"{name}: {start:,}/{len(data):,}", flush=True)
    if labels != data["label"].tolist() or not math.isfinite(loss_sum):
        raise ValueError(f"{name} labels or loss are invalid.")
    return loss_sum / len(labels), predictions, probabilities, truncated

validation_data = pd.read_csv(validation_path)
validation_loss, validation_predictions, _, _ = predict(validation_data, "Validation")
validation_metrics = classification_metrics(validation_data["label"], validation_predictions)
if not math.isclose(validation_loss, selected_row["VALIDATION LOSS"], abs_tol=1e-6) or any(
    not math.isclose(value, selected_row[metric], abs_tol=1e-12) for metric, value in validation_metrics.items()
):
    raise ValueError("Checkpoint validation results do not reproduce the selected epoch.")
print("Selected checkpoint validation metrics reproduced.")


In [ ]:
# 使用同一测试集，对齐样本顺序并保存扩展对照结果
test_data = pd.read_csv(test_path)
prior_predictions = pd.read_csv(prior_predictions_path, usecols=["TEST ROW", "LABEL"])
if (test_data.empty or test_data["text"].isna().any() or not test_data["label"].isin([0, 1]).all()
        or prior_predictions["TEST ROW"].tolist() != list(range(len(test_data)))
        or prior_predictions["LABEL"].tolist() != test_data["label"].tolist()):
    raise ValueError("Test rows or labels do not match the previous evaluation.")
test_loss, test_predictions, test_probabilities, truncated = predict(test_data, "Test")
test_metrics = classification_metrics(test_data["label"], test_predictions)
tn, fp, fn, tp = confusion_matrix(test_data["label"], test_predictions, labels=[0, 1]).ravel()

comparison = pd.read_csv(prior_results_path)
row = {
    "MODEL": "DistilBERT", "ROLE": "Pretrained extension",
    "CONFIGURATION": f"Seed {protocol['seed']}, epoch {protocol['epoch']}, max length {protocol['max_length']}",
}
for metric in validation_metrics:
    row["VALIDATION " + metric] = validation_metrics[metric]
    row["TEST " + metric] = test_metrics[metric]
comparison = pd.concat([comparison, pd.DataFrame([row])], ignore_index=True)

predictions = pd.DataFrame({
    "TEST ROW": range(len(test_data)), "LABEL": test_data["label"],
    "DISTILBERT POSITIVE PROBABILITY": test_probabilities, "DistilBERT": test_predictions,
})
matrix = pd.DataFrame([{"MODEL": "DistilBERT", "TN": tn, "FP": fp, "FN": fn, "TP": tp}])
predictions.to_csv(prediction_path, index=False)
matrix.to_csv(matrix_path, index=False)
comparison.to_csv(comparison_path, index=False)
display(comparison[["MODEL", "VALIDATION F1", "TEST ACCURACY", "TEST PRECISION", "TEST RECALL", "TEST F1"]].round(4))
display(matrix)
print(f"Test rows: {len(test_data):,} | loss: {test_loss:.4f} | truncated: {truncated:,} ({truncated / len(test_data):.2%})")


## 结果

固定的第 1 轮检查点在 24,678 条测试评论上取得 Accuracy 0.9072、Precision 0.8964、Recall 0.9220、F1 0.9090。B06 与 CNN + 单向 LSTM 等权融合的测试 F1 分别为 0.8901 和 0.8704。DistilBERT 的混淆矩阵为 TN 10,943、FP 1,323、FN 968、TP 11,444。

按 256 词项上限，测试集有 9,906 条评论被截断（40.14%）。本实验使用种子 42 的单次微调，模型之间同时存在外部预训练、分词和输入设置的差异。项目中其他方案的测试指标在本次扩展实验之前已查看。
